In [9]:
import pandas as pd


In [10]:
# =========================================================
# 1. PATH SETUP
# =========================================================
FLOW_FILE           = "../data/intermediate/park_flows_placekey.csv"
AUDIT_FILE          = "../data/output/parks_audit.csv"
CENTROID_FILE       = "../data/intermediate/park_property_centroids.csv"
PROP_PLACEKEYS_FILE = "../data/intermediate/property_placekeys.csv"
TRACT_SHP_PATH      = "../data/raw/census/tl_2023_36_tract.shp"
OUTPUT_FILE         = "../data/output/park_flows_property_10km.csv"
MAX_DIST_KM         = 10.0


In [11]:
# =========================================================
# 2. LOAD INPUTS
# =========================================================
import geopandas as gpd
import pandas as pd
import numpy as np
import ast

print("Loading flow data...")
flow_df = pd.read_csv(FLOW_FILE)
print(f"Flow rows: {len(flow_df)}")

print("Loading audit table...")
audit = pd.read_csv(AUDIT_FILE)
pk_to_prop = audit[['placekey', 'parent_placekey', 'gis_prop_num']].dropna(subset=['gis_prop_num'])

prop_names = (
    audit[['gis_prop_num', 'property_name']]
    .dropna(subset=['gis_prop_num'])
    .drop_duplicates(subset='gis_prop_num')
)

print("Loading property -> placekeys lookup...")
prop_placekeys = pd.read_csv(PROP_PLACEKEYS_FILE)
prop_placekeys['all_placekeys'] = prop_placekeys['all_placekeys'].apply(ast.literal_eval)
print(f"Properties in lookup: {len(prop_placekeys)}")

print("Loading property centroids...")
centroids = pd.read_csv(CENTROID_FILE)
print(f"Property centroids: {len(centroids)}")

print("Loading NYC tract shapefile...")
nyc_counties = ['005', '047', '061', '081', '085']
tracts = gpd.read_file(TRACT_SHP_PATH)
nyc_tracts = tracts[tracts['COUNTYFP'].isin(nyc_counties)].copy().to_crs('EPSG:2263')
nyc_tracts['tract_centroid_x'] = nyc_tracts.geometry.centroid.x
nyc_tracts['tract_centroid_y'] = nyc_tracts.geometry.centroid.y
tract_centroids = nyc_tracts[['GEOID', 'tract_centroid_x', 'tract_centroid_y']].rename(
    columns={'GEOID': 'tract_i'}
)
print(f"Tract centroids: {len(tract_centroids)}")


Loading flow data...
Flow rows: 216367
Loading audit table...
Loading property -> placekeys lookup...
Properties in lookup: 1022
Loading property centroids...
Property centroids: 2056
Loading NYC tract shapefile...
Tract centroids: 2327


In [12]:
# =========================================================
# 3. MAP PLACEKEYS TO PROPERTIES
# =========================================================
flow_df = flow_df.merge(pk_to_prop, left_on='park_j', right_on='placekey', how='left')
print(f"Matched: {flow_df['gis_prop_num'].notna().sum()}/{len(flow_df)} rows")

Matched: 216367/216367 rows


In [13]:
# =========================================================
# 4. IDENTIFY QUALIFYING TRACT-PROPERTY PAIRS
# A pair qualifies if ANY POI in the property is within 10km
# =========================================================
qualifying_pairs = (
    flow_df.groupby(['tract_i', 'gis_prop_num'])['distance_km']
    .min()
    .reset_index()
    .rename(columns={'distance_km': 'min_poi_distance_km'})
)
qualifying_pairs = qualifying_pairs[
    qualifying_pairs['min_poi_distance_km'] <= MAX_DIST_KM
][['tract_i', 'gis_prop_num']]
print(f"Qualifying tract-property pairs: {len(qualifying_pairs)}")


Qualifying tract-property pairs: 100851


In [14]:
# =========================================================
# 5. AGGREGATE VISITS FOR QUALIFYING PAIRS
# =========================================================
print("Aggregating visits to property level...")
flow_filtered = flow_df.merge(qualifying_pairs, on=['tract_i', 'gis_prop_num'], how='inner')

property_flows = (
    flow_filtered.groupby(['tract_i', 'gis_prop_num'])
    .agg(visits=('visits', 'max'))
    .reset_index()
)

# Attach all placekeys from audit-based lookup
property_flows = property_flows.merge(prop_placekeys, on='gis_prop_num', how='left')

# Attach official property name
property_flows = property_flows.merge(prop_names, on='gis_prop_num', how='left')
print(f"Property-level rows: {len(property_flows)}")


Aggregating visits to property level...
Property-level rows: 100851


In [15]:
# =========================================================
# 6. COMPUTE DISTANCE — tract centroid to property centroid
# =========================================================
print("Computing centroid-to-centroid distances...")

property_flows['tract_i'] = property_flows['tract_i'].astype(str)
tract_centroids['tract_i'] = tract_centroids['tract_i'].astype(str)

property_flows = property_flows.merge(tract_centroids, on='tract_i', how='left')
property_flows = property_flows.merge(centroids, on='gis_prop_num', how='left')

property_flows['distance_km'] = (
    np.sqrt(
        (property_flows['tract_centroid_x'] - property_flows['centroid_x']) ** 2 +
        (property_flows['tract_centroid_y'] - property_flows['centroid_y']) ** 2
    ) * 0.3048 / 1000
).round(3)

property_flows = property_flows.drop(columns=[
    'tract_centroid_x', 'tract_centroid_y', 'centroid_x', 'centroid_y'
])

property_flows = property_flows[[
    'tract_i', 'gis_prop_num', 'property_name',
    'visits', 'distance_km', 'all_placekeys'
]]

property_flows = property_flows.sort_values(
    ['tract_i', 'distance_km', 'gis_prop_num']
).reset_index(drop=True)

print(f"Final rows: {len(property_flows)}")
property_flows.head()


Computing centroid-to-centroid distances...


Final rows: 100851


,tract_i,gis_prop_num,property_name,visits,distance_km,all_placekeys
0,36005000100,Q332,Woodtree Playground,8,2.315,"[zzy-223@627-sb4-p5f, zzz-222@627-sb5-8jv]"
1,36005000100,X138,Playground One Thirty Four CXXXIV,14,3.092,[zzy-223@627-sbf-q4v]
2,36005000100,Q366,Louis C. Moser Playground,4,3.369,[zzz-222@627-sb6-j5f]
3,36005000100,Q004,Astoria Park,46,3.699,"[226-222@627-sb7-wzf, zzy-222@627-sb8-kfz, zzy..."
4,36005000100,M108B,Park,9,3.942,[zzy-222@627-sbf-z9f]


In [16]:
# =========================================================
# 7. SAVE
# =========================================================
property_flows.to_csv(OUTPUT_FILE, index=False)
print(f"Saved to: {OUTPUT_FILE}")


Saved to: ../data/output/park_flows_property_10km.csv
